# Coevaluación de equipos en Google Colab

Este notebook prepara Flask + Angular y publica la aplicación mediante una URL HTTPS temporal de ngrok. Activa un acceso de demostración sin Google: no introduzcas datos reales. Consulta `docs/COLAB.md` antes de ejecutarlo.

In [ ]:
# Si se activa, la base SQLite sobrevivirá a los reinicios del entorno.
USAR_GOOGLE_DRIVE = False
RAMA = "colab"

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path("/content/coevaluacion-equipos")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", RAMA,
        "https://github.com/anderfrago/coevaluiacion-equipos.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
print(f"Código preparado en {REPO}")

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

# Angular 22 necesita una versión reciente de Node. `n` evita depender de la versión preinstalada en Colab.
subprocess.run(["npm", "install", "-g", "n"], check=True)
subprocess.run(["n", "22.22.3"], check=True)
subprocess.run(["/usr/local/bin/npm", "ci"], cwd=REPO / "frontend", check=True)
subprocess.run(["/usr/local/bin/npm", "run", "build"], cwd=REPO / "frontend", check=True)
print("Dependencias instaladas y frontend compilado.")

In [ ]:
from google.colab import userdata
import secrets

def secreto(nombre, obligatorio=True, predeterminado=""):
    try:
        valor = userdata.get(nombre)
    except Exception:
        valor = None
    valor = (valor or predeterminado).strip()
    if obligatorio and not valor:
        raise RuntimeError(f"Falta el secreto de Colab {nombre}.")
    return valor

configuracion = {
    "SECRET_KEY": secrets.token_urlsafe(48),
    "BASE_URL": "https://colab-demo.invalid",
    "COOKIE_SECURE": "true",
    "COLAB_DEMO_MODE": "true",
}

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/coevaluacion-colab")
else:
    data_dir = Path("/content/coevaluacion-data")
data_dir.mkdir(parents=True, exist_ok=True)
configuracion["DATABASE_URL"] = f"sqlite:///{data_dir / 'coevaluacion.db'}"
os.environ.update(configuracion)
print("Configuración de demostración preparada.")

In [ ]:
import threading
import time
from pyngrok import ngrok
from waitress import serve
from wsgi import application
from backend.models import db

with application.app_context():
    db.create_all()

servidor = threading.Thread(
    target=serve, kwargs={"app": application, "host": "127.0.0.1", "port": 5000}, daemon=True
)
servidor.start()
time.sleep(2)

ngrok.set_auth_token(secreto("NGROK_AUTHTOKEN"))
tunel = ngrok.connect(5000, bind_tls=True)
public_url = tunel.public_url.rstrip("/")
application.config["BASE_URL"] = public_url
print(f"Aplicación disponible en: {public_url}")
print("Cualquiera con esta URL tendrá acceso de administrador: no uses datos reales.")
print("Mantén esta celda y el entorno de Colab activos.")